# Detekce čerpacích cyklů a kontrola fungování čerpadla

Analýza souboru `Pressure.csv` dle zadání popsaném v README.adoc: definice cyklů, prahové hodnoty (A), hypotéza trendu tlaku během cyklu (B).

In [9]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
pd.options.display.float_format = "{:.4f}".format

df = pd.read_csv("Pressure.csv", parse_dates=["Date"]).sort_values("Date").set_index("Date")

display(Markdown("## Data Preview"))
display(df.head(10))

#display(Markdown("## Summary Statistics"))
#display(df.describe())

display(Markdown("## Missing Values"))
display(df.isna().sum().to_frame("missing_count"))

#display(Markdown("## Data Info"))
#df.info()

## Data Preview

,Int,Pressure,PumpState
Date,,,
2017-08-13 00:27:40.050,0,1.7308,NaN
2017-08-13 02:10:00.500,0,1.5434,NaN
2017-08-13 03:01:49.500,0,NaN,16.0000
2017-08-13 03:02:08.000,0,1.3728,NaN
2017-08-13 03:02:15.500,0,NaN,32.0000
2017-08-13 03:02:58.000,0,1.5434,NaN
2017-08-13 03:35:33.590,0,NaN,1.0000
2017-08-13 03:35:33.880,0,1368.3842,NaN
2017-08-13 03:35:34.170,0,9133.9434,NaN


## Missing Values

,missing_count
Int,0
Pressure,2814
PumpState,19150


## Stav čerpadla (`PumpState`)

V CSV je `PumpState` vyplněn jen při **změně** stavu, jinde je `NaN`. Níže jsou všechny hodnoty, které se v datech vyskytují, a kolikrát byly **zalogovány** (počet řádků s neprázdným `PumpState`).

In [10]:
pump_counts = (
    df["PumpState"]
    .value_counts(dropna=False)
    .rename_axis("PumpState")
    .to_frame("count")
)

pump_counts.sort_index()

,count
PumpState,
0.0000,31
1.0000,153
2.0000,189
16.0000,154
32.0000,185
NaN,19150


# Pumping Cycle Algoritmus
- Algoritmus definuje validní čerpací cykly na základě změn hodnoty `PumpState`
- Cyklus začíná 10s po zalogování změny stavu na hodnotu `(1)`
- Cyklus končí 5s před zalogování změny stavu na hodnotu `(2)`
- Během cyklu `PumpState` zůstává na hodnotě `(1)` a nesmí se změnit na hodnotu jinou.
- V rámci cyklu se mohou vyskytovat další logy hodnoty `(1)`

## Edge cases
- Výskyt jiného stavu než `(1)` mezi `(1)` a `(2)` zneplatní cyklus
- Pokud nenásleduje stav `(2)`, cyklus není validní
- Po aplikaci posunů `+10s` a `-5s` může vzniknout prázdný interval
- Interval nemusí obsahovat žádné měření `Pressure` kvůli chybějícím nebo řídkým datům

In [11]:
# Cycle Detection Function
import numpy as np
from scipy import stats

# Strict numeric checks: PumpState in the assignment is numeric (1=pumping,2=pumped)
def _is_state_pumping(v):
    try:
        return int(v) == 1
    except (ValueError, TypeError):
        return False

def _is_state_pumped(v):
    try:
        return int(v) == 2
    except (ValueError, TypeError):
        return False


def detect_cycles(df, pressure_col='Pressure', state_col='PumpState',
                  start_offset=pd.Timedelta('10s'), end_offset=pd.Timedelta('5s')):
    """Detect pumping cycles following the README specification:
    - cycle starts `start_offset` after a logged PumpState == 1
    - cycle ends `end_offset` before a logged PumpState == 2
    - cycle is valid only if forward-filled PumpState remains as 1 for whole interval
    """
    df2 = df.copy()
    # forward-fill the pump state so each timestamp has the current state
    df2['_pump_ff'] = df[state_col].ffill()

    # events are rows where PumpState was explicitly logged
    events = df[state_col].dropna()
    starts = (events[events.apply(_is_state_pumping)].index + start_offset).tolist()
    ends = (events[events.apply(_is_state_pumped)].index - end_offset).tolist()

    starts.sort()
    ends.sort()

    cycles = []
    # index for ends
    ei = 0 
    for s in starts:
        # find first end after start
        while ei < len(ends) and ends[ei] <= s:
            ei += 1
        if ei >= len(ends):
            print(f"Warning: no more cycle ends after start at {s}, stopping detection")
            break
        e = ends[ei]
        # check that forward-filled PumpState is consistently 1 in [s,e]
        seg_state = df2['_pump_ff'].loc[s:e]
        if seg_state.empty:
            print(f"Warning: no data points between {s} and {e}, skipping cycle")
            continue
        if not seg_state.apply(_is_state_pumping).all():
            print(f"Warning: PumpState is not consistently 1 between {s} and {e}, skipping cycle")
            continue
        seg = df.loc[s:e]
        dur = e - s
        # compute trend if enough points
        slope = np.nan
        if len(seg) >= 2:
            x = (seg.index.view('int64') - seg.index.view('int64')[0]) / 1e9
            y = seg[pressure_col].values
            slope, intercept, r, p, se = stats.linregress(x, y)[:5]
        cycles.append({
            'start': s,
            'end': e,
            'duration_seconds': dur.total_seconds(),
            'n_points': len(seg),
            'pressure_min': seg[pressure_col].min() if len(seg) else np.nan,
            'pressure_max': seg[pressure_col].max() if len(seg) else np.nan,
            'pressure_mean': seg[pressure_col].mean() if len(seg) else np.nan,
            'trend_slope': slope,
        })
        ei += 1

    return pd.DataFrame(cycles)

# Run detection
cycles = detect_cycles(df)

# Display results
display(Markdown('## Detekované cykly'))
display(cycles)

## Detekované cykly

,start,end,duration_seconds,n_points,pressure_min,pressure_max,pressure_mean,trend_slope
0,2017-08-13 03:35:43.590,2017-08-13 03:37:13.630,90.0400,85,8.2568,7000.5081,491.0286,-24.4300
1,2017-08-13 07:09:15.170,2017-08-13 07:10:55.160,99.9900,84,7.9050,7000.5081,385.1872,-16.7667
2,2017-08-13 07:39:33.300,2017-08-13 07:40:30.280,56.9800,84,8.9999,6051.5248,399.1156,-35.0280
3,2017-08-13 11:14:40.130,2017-08-13 11:16:03.140,83.0100,85,8.6216,6051.5248,418.4055,-22.1712
4,2017-08-13 12:00:16.420,2017-08-13 12:01:01.450,45.0300,82,9.2596,7265.5804,502.2524,-53.7665
...,...,...,...,...,...,...,...,...
148,2017-09-11 00:01:49.110,2017-09-11 00:03:06.090,76.9800,82,9.7980,7000.5081,482.0164,-30.7068
149,2017-09-11 03:19:41.220,2017-09-11 03:20:51.200,69.9800,77,9.5256,6273.0914,462.7924,-29.2089
150,2017-09-11 10:26:09.410,2017-09-11 10:27:52.450,103.0400,86,9.1290,6273.0914,418.5258,-19.0362
151,2017-09-11 19:21:36.160,2017-09-11 19:23:23.160,107.0000,84,9.1290,7832.9396,534.3607,-24.1516
